In [ ]:
#!/usr/bin/env python3
"""Loads the inputs of the pipeline: the query table, the downloaded or copied protein sequences and the UniProt annotation fields of every label.

Paths come from the dw_uniprot_seqs and dw_uniprot_features sections of configs/param_configs.yaml; labels are named and ordered as in the query table (exact_species). Local FASTA sources have sequences but no UniProt tables.
"""
###################################### DEPENDENCIES_AND_CONSTANTS
from IPython.display import display
import pandas as pd
from pathlib import Path
from pyprojroot import find_root, has_file
import yaml
DEFAULT_QUERY_TSV = "configs/query_configs.tsv"
YAML_PATH = "configs/param_configs.yaml"
###################################### CONFIG
PROJECT_ROOT = find_root(has_file(".here"), start=Path.cwd()).resolve()
config = yaml.safe_load((PROJECT_ROOT / YAML_PATH).read_text()) or {}
query_table = pd.read_csv(PROJECT_ROOT / config.get("dw_uniprot_seqs", {}).get("query_table", DEFAULT_QUERY_TSV), sep="\t", dtype=str)
species_names = dict(zip(query_table["label"], query_table["exact_species"].fillna(query_table["label"])))
sequences_dir = PROJECT_ROOT / config["dw_uniprot_seqs"]["sequences_dir"]
features_dir = PROJECT_ROOT / config["dw_uniprot_features"]["features_dir"]
display(query_table)

In [ ]:
###################################### LOAD
def fasta_lengths(path: Path) -> pd.Series:
    """Read sequence lengths from a FASTA file.

    Args:
        path (Path): FASTA file.

    Returns:
        pd.Series: Length per sequence id.
    """
    lengths, seq_id = {}, None
    with open(path) as f:
        for line in f:
            if line.startswith(">"): seq_id = line[1:].split()[0]; lengths[seq_id] = 0
            elif seq_id: lengths[seq_id] += len(line.strip())
    return pd.Series(lengths, name="length")
seq_lengths = {lbl: fasta_lengths(sequences_dir / f"{lbl}_seqs.fasta") for lbl in species_names if (sequences_dir / f"{lbl}_seqs.fasta").exists()}
sequence_tables = {lbl: pd.read_csv(sequences_dir / f"{lbl}_seqs.tsv", sep="\t") for lbl in species_names if (sequences_dir / f"{lbl}_seqs.tsv").exists()}
feature_tables = {lbl: pd.read_csv(features_dir / f"{lbl}_feats.tsv", sep="\t", dtype=str, low_memory=False) for lbl in species_names if (features_dir / f"{lbl}_feats.tsv").exists()}
summary = pd.DataFrame({"species": species_names, "FASTA sequences": {l: len(s) for l, s in seq_lengths.items()}, "median length": {l: s.median() for l, s in seq_lengths.items()}, "UniProt entries": {l: len(t) for l, t in sequence_tables.items()}, "annotation fields": {l: t.shape[1] for l, t in feature_tables.items()}})
display(summary)